# Chronos 2 Modelling

## Load the dataset

In [1]:
import pandas as pd
df_full = pd.read_parquet("../Data/EastAfrica Preprocessed/eastafrica.parquet")

## Type conversion and minor preprocessing

In [2]:
print("Shape: ", df_full.shape)

for col in df_full.columns:
    print (f"{col}: ", df_full[col].dtype)

Shape:  (2314906, 55)
id:  category
name:  category
country:  category
latitude:  float64
longitude:  float64
minBorderDistanceKm:  float64
minCapitalDistanceKm:  float64
periodStart:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float64
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float64
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPolitic

### Type Conversion

In [3]:
#convert float64s to float32 to save memory:
float_cols = df_full.select_dtypes(include=["float64"]).columns

df_full[float_cols] = df_full[float_cols].astype("float32")


In [4]:
#Convert id to string; if it's a category issues have been seen with categories being deleted that are still referenced with 0 observations
df_full["id"] = df_full["id"].astype("string")

### Apply Target Variables

In [5]:
#Local:
df_full['target_conflict_indicator_local'] = (
    (df_full['LocalTotalFatalities'] >= 3) |
    (df_full['LocalMeanSeverity'] >= 3)
).astype(int)

#Regional:
df_full['target_conflict_indicator_regional'] = (
    (df_full['RegionalTotalFatalities'] >= 3) |
    (df_full['RegionalMeanSeverity'] >= 3)
).astype(int)

target_columns = ['target_conflict_indicator_local', 'target_conflict_indicator_regional']

In [6]:
print("Shape: ", df_full.shape)

for col in df_full.columns:
    print (f"{col}: ", df_full[col].dtype)

Shape:  (2314906, 57)
id:  string
name:  category
country:  category
latitude:  float32
longitude:  float32
minBorderDistanceKm:  float32
minCapitalDistanceKm:  float32
periodStart:  datetime64[ns]
LocalEventCount:  int64
LocalDistinctActorCount:  int64
LocalDistinctEventTypes:  int64
LocalDistinctEventSubtypes:  int64
LocalTotalFatalities:  int64
LocalMeanSeverity:  float32
LocalMaxSeverity:  int64
LocalProtestersEventCount:  int64
LocalStateForcesEventCount:  int64
LocalPoliticalMilitiaEventCount:  int64
LocalIdentityMilitiaEventCount:  int64
LocalRebelGroupEventCount:  int64
LocalRiotersEventCount:  int64
LocalCiviliansEventCount:  int64
LocalOtherTypeEventCount:  int64
RegionalEventCount:  int64
RegionalDistinctActorCount:  int64
RegionalDistinctEventTypes:  int64
RegionalDistinctEventSubTypes:  int64
RegionalTotalFatalities:  int64
RegionalMeanSeverity:  float32
RegionalMaxSeverity:  int64
RegionalProtestersEventCount:  int64
RegionalStateForcesEventCount:  int64
RegionalPolitical

In [7]:
#Drop columns named max: suspect leakage.
df_full = df_full.drop(columns=df_full.filter(regex="Max").columns)

#drop the median columns if they still somehow exist:
df_full = df_full.drop(columns=df_full.filter(regex="Median").columns)
df_full = df_full.drop(columns=df_full.filter(regex="median").columns)

### Remove rows with missing data:

In [8]:
df_full = df_full.dropna(
    subset=["Armed_forces_personnel__of_total_labor_force_MSMILTOTLTFZS"]
)

In [9]:
df_full.head()

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,LocalEventCount,LocalDistinctActorCount,...,Armed_forces_personnel__of_total_labor_force_MSMILTOTLTFZS,Military_expenditure__of_GDP_MSMILXPNDGDZS,Renewable_internal_freshwater_resources_per_capita_cubic_meters_ERH2OINTRPC,Cereal_yield_kg_per_hectare_AGYLDCRELKG,Permanent_cropland__of_land_area_AGLNDCROPZS,Water_productivity_total_constant_2015_US_GDP_per_cubic_meter_of_total_freshwater_withdrawal_ERGDPFWTLM3KD,Internally_displaced_persons_new_displacement_associated_with_conflict_and_violence_number_of_cases_VCIDPNWCV,is_voting,target_conflict_indicator_local,target_conflict_indicator_regional
241612,Central_African_Republic_Aba,Aba,Central African Republic,6.1054,15.314900,61.576637,411.333405,2016-01-28,0,0,...,0.461758,1.471638,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0
239535,Central_African_Republic_Abagba_2,Abagba 2,Central African Republic,6.4194,21.998699,204.220184,442.083832,2016-01-28,0,0,...,0.461758,1.471638,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0
236873,Central_African_Republic_Abba,Abba,Central African Republic,5.4166,15.166600,60.871994,397.073151,2016-01-28,0,0,...,0.461758,1.471638,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0
248130,Central_African_Republic_Abba-Bogani,Abba-Bogani,Central African Republic,5.8296,15.519500,97.355522,377.276337,2016-01-28,0,0,...,0.461758,1.471638,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0
238249,Central_African_Republic_Agoudou-Manga,Agoudou-Manga,Central African Republic,5.8823,20.912600,139.503586,308.460327,2016-01-28,0,0,...,0.461758,1.471638,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0


## Dataset Validation

In [10]:

df_counts = (
    df_full.groupby("id", observed = False)
      .size()
      .reset_index(name="count")
)




bad_ids = df_counts.query("count != 125")

print("Bad IDs in df_full:", bad_ids["id"].tolist())

# Show counts for each bad ID
print("Counts for bad IDs (df_full):")
print(bad_ids)


Bad IDs in df_full: ['Kenya_Busia', 'Kenya_Gatare', 'Kenya_Mbale', 'Sudan_Northern', 'Uganda_Kigoma', 'Uganda_Rubirizi']
Counts for bad IDs (df_full):
                    id  count
4692       Kenya_Busia    375
5016      Kenya_Gatare    375
6476       Kenya_Mbale    375
13904   Sudan_Northern    375
15578    Uganda_Kigoma    375
16500  Uganda_Rubirizi    375


In [11]:
# compute counts
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)

# get list of bad IDs
bad_ids = df_counts.query("count != 125")["id"].tolist()

# drop rows with those IDs
print("Before deletion:", df_full[df_full["id"].isin(bad_ids)].shape)
df_full.drop(df_full[df_full["id"].isin(bad_ids)].index, inplace=True)
print("After deletion:", df_full[df_full["id"].isin(bad_ids)].shape)


# recompute counts to verify
df_counts = (
    df_full.groupby("id", observed=False)
      .size()
      .reset_index(name="count")
)
df_full.reset_index(drop=True, inplace=True)


bad_ids_after = df_counts.query("count != 125")
print("Bad IDs in df_full:", bad_ids_after["id"].tolist())
print("Counts for bad IDs (df_full):")
print(bad_ids_after)


Before deletion: (2250, 55)
After deletion: (0, 55)
Bad IDs in df_full: []
Counts for bad IDs (df_full):
Empty DataFrame
Columns: [id, count]
Index: []


## Build the Test and Train datasets. 

In [12]:
all_period_starts = sorted(df_full['periodStart'].unique(), reverse = True)
all_period_starts
testing_periods = all_period_starts[:12]
testing_periods

training_periods = [p for p in all_period_starts if p not in testing_periods]
print("Training Periods: ", min(training_periods), ' - ', max(training_periods))
print("Testing Periods: ", min(testing_periods), ' - ', max(testing_periods))

Training Periods:  2016-01-28 00:00:00  -  2024-08-29 00:00:00
Testing Periods:  2024-09-26 00:00:00  -  2025-07-31 00:00:00


In [13]:
df_full = df_full.sort_values(["id", "periodStart"])

In [14]:
import pandas as pd
from datetime import datetime

def build_context_dataset(
    df: pd.DataFrame,
    cutoffDate: datetime | None = None,
    ids: str | None = None,
    n_high: int = 50,
    n_low: int = 25,
    n_spike: int = 25,
    n_transition: int = 25,
    n_future_obs: int = 12
):
    
    df_source = df.copy()
    
    specific_ids = []
    
    if ids is not None:
        if isinstance(ids, list):
            # Normalize each ID in the list
            specific_ids = [
                x.strip().replace(" ", "_")
                for x in ids
                if isinstance(x, str) and x.strip() != ""
            ]
        else:
            raise TypeError("ids must be either a list of strings")

    
    # Use the new indicator columns as the conflict signal
    # Assuming they are 0/1 or boolean; cast to int to be safe.
    df_source["conflict_score"] = (
        df_source["target_conflict_indicator_local"].astype(int)
        + df_source["target_conflict_indicator_regional"].astype(int)
    )

    # Aggregate by id (place)
    agg = (
        df_source.groupby("id", observed = False)
        .agg(
            total_conflict=("conflict_score", "sum"),
            max_spike=("conflict_score", lambda s: s.diff().abs().max()),
        )
        .reset_index()
    )

    # --- High conflict places -------------------------------------------------
    high_ids = (
        agg.sort_values("total_conflict", ascending=False)
        .head(n_high)["id"]
        .tolist()
    )

    # --- Low conflict places --------------------------------------------------
    low_ids = (
        agg.sort_values("total_conflict", ascending=True)
        .head(n_low)["id"]
        .tolist()
    )

    # --- Spike-based places ---------------------------------------------------
    spike_ids = (
        agg.sort_values("max_spike", ascending=False)
        .head(n_spike)["id"]
        .tolist()
    )

    #detect transitions: any change from previous timestep
    df_source["transition_flag"] = (
        df_source.groupby("id")["conflict_score"]
        .diff()
        .abs()
        > 0
    ).astype(int)
    
    #count transitions per id
    transition_ids = (
        df_source.groupby("id")["transition_flag"]
        .sum()
        .sort_values(ascending=False)
        .head(n_transition)
        .index
        .tolist()
    )
    
    # Combine unique ids
    selected_ids = (
        set(specific_ids)
        | set(high_ids)
        | set(low_ids)
        | set(spike_ids)
        | set(transition_ids)
    )

    #extract full windows for each selected id
    df_context = df_source[df_source["id"].isin(selected_ids)].copy().drop_duplicates()


    
    #dynamically build out the future set:
    df_future_columns =  ["id", "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm",	"periodStart", 'is_voting',  'target_conflict_indicator_local', 'target_conflict_indicator_regional'] #these are covariates we can know at the time of prediction. We also output the targets: we'll remove them when actually doing the training, but useful for evaluation.
    df_future = df_source[df_source['periodStart'] > cutoffDate][df_future_columns].copy()

    
    #if we do specify a specific id(s), apply dummy values to the other ids in df_future:
    if specific_ids or len(ids) > 0:
        # IDs we do NOT care about
        other_ids = set(df_future["id"].unique()) - set(specific_ids)

        # Overwrite unwanted IDs with defaults
        for oid in other_ids:
            mask = df_future["id"] == oid

            # Overwrite all future covariates with safe defaults
            df_future.loc[mask, "name"] = None
            df_future.loc[mask, "country"] = None
            df_future.loc[mask, "latitude"] = 0.0
            df_future.loc[mask, "longitude"] = 0.0
            df_future.loc[mask, "minBorderDistanceKm"] = 0.0
            df_future.loc[mask, "minCapitalDistanceKm"] = 0.0
            df_future.loc[mask, "is_voting"] = False
    
    #now use cutoffDate to remove future records from the testing set:
    df_context = df_context[df_context['periodStart'] <= cutoffDate]   

    print(min(df_context['periodStart']))
    print(max(df_context['periodStart']))
    print(min(df_future['periodStart']))
    print(max(df_future['periodStart']))
    
    #validate: every id must have exactly n_future_obs observations
    counts = df_future.groupby("id", observed = True).size()
    invalid_ids = counts[counts < n_future_obs]
    if len(invalid_ids) > 0 or  df_future.empty:
        raise ValueError(f"Invalid future window: the following ids do not have 12 observations:\n{invalid_ids}. Try setting the cutoff date to an earlier value.")

    #remove extra future records:
    df_future = (
        df_future.groupby("id")
        .head(12)
        .reset_index(drop=True)
    )
    
    return df_context.sort_values(["id", "periodStart"]), df_future.sort_values(["id", "periodStart"])

In [15]:
#df_context, df_future = build_context_dataset(df_source = df, cutoffDate = '2023-08-01', ids=["Sudan_Omdurman_-_Al_Doha"] )
cutoffDate = '2024-08-29'
df_context, df_future = build_context_dataset(df = df_full, cutoffDate = cutoffDate, ids=[])

df_future = df_future[df_future["id"].isin(df_context["id"].unique())].copy()

#print(df_context)
print(df_context.groupby("id", observed = False).size().reset_index(name="count"))
print("shape: ", df_context.shape)
print(sorted(df_context['id'].unique()))

#print(df_context)
print(df_future.groupby("id", observed = True).size().reset_index(name="count"))
print("shape: ", df_future.shape)
print(sorted(df_future['id'].unique()))




2016-01-28 00:00:00
2024-08-29 00:00:00
2024-09-26 00:00:00
2025-07-31 00:00:00
                                     id  count
0      Central_African_Republic_Bambari    113
1    Central_African_Republic_Dangbatro    113
2       Central_African_Republic_Pakale    113
3                            Chad_Pende    113
4                             Chad_Sido    113
..                                  ...    ...
120                     Sudan_Suni_East    113
121                        Uganda_Alero    113
122                        Uganda_Lango    113
123                       Uganda_Lokori    113
124           Uganda_Zombo_Town_Council    113

[125 rows x 2 columns]
shape:  (14125, 57)
['Central_African_Republic_Bambari', 'Central_African_Republic_Dangbatro', 'Central_African_Republic_Pakale', 'Chad_Pende', 'Chad_Sido', 'Egypt_Abu_Hamad', 'Egypt_Abu_Simbel', 'Egypt_Al_Amiriyah', 'Egypt_Badr', 'Ethiopia_Pagak', 'Kenya_Bukwo', 'Kenya_Chari', 'Kenya_Elwak_North', 'Kenya_Kapsaos', 'Kenya_Kapsenge

In [16]:
#Data leakage check: check no dates after the cutoff in the context dataset. 
print(df_context[df_context["periodStart"] > cutoffDate])



Empty DataFrame
Columns: [id, name, country, latitude, longitude, minBorderDistanceKm, minCapitalDistanceKm, periodStart, LocalEventCount, LocalDistinctActorCount, LocalDistinctEventTypes, LocalDistinctEventSubtypes, LocalTotalFatalities, LocalMeanSeverity, LocalProtestersEventCount, LocalStateForcesEventCount, LocalPoliticalMilitiaEventCount, LocalIdentityMilitiaEventCount, LocalRebelGroupEventCount, LocalRiotersEventCount, LocalCiviliansEventCount, LocalOtherTypeEventCount, RegionalEventCount, RegionalDistinctActorCount, RegionalDistinctEventTypes, RegionalDistinctEventSubTypes, RegionalTotalFatalities, RegionalMeanSeverity, RegionalProtestersEventCount, RegionalStateForcesEventCount, RegionalPoliticalMilitiaEventCount, RegionalIdentityMilitiaEventCount, RegionalCiviliansEventCount, RegionalRebelGroupEventCount, RegionalRiotersEventCount, RegionalOtherTypeEventCount, month_sin, month_cos, Population_total_SPPOPTOTL, Control_of_Corruption__Governance_estimate_approx_25_to_+25_GOV_WGI_

## Raw Model Modelling

In [174]:
#Install the amazon chronos package for chronos 2
!pip install chronos-forecasting

from chronos import Chronos2Pipeline

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: restrict to one GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "0"



vanilla_chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [175]:
df_context = df_context.sort_values(["id", "periodStart"])
df_future = df_future.sort_values(["id", "periodStart"])

### Local conflict (Vanilla)

In [176]:
# Generate predictions with covariates: takes roughly 4 mins. 
import datetime;
import joblib

ct = datetime.datetime.now()
print("current time:", ct)


pred_df_local = vanilla_chronos2_pipeline.predict_df(
    df =df_context.drop(columns=["target_conflict_indicator_regional"]), #important to drop the other target, they are strong but not perfect indicators of the other. ,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(), #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_local",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_local

ct = datetime.datetime.now()
print("current time:", ct)

joblib.dump(vanilla_chronos2_pipeline, f"../Models/Chronos2/chronos2_model_local_vanilla.pkl") #save the full model as a pickle file. 

current time: 2026-10-10 16:31:27.698070
current time: 2026-10-10 16:32:12.320858


['../Models/Chronos2/chronos2_model_local_vanilla.pkl']

### Regional conflict (Vanilla)

In [177]:
# Generate predictions with covariates: takes roughly 4 mins. 

import datetime;

ct = datetime.datetime.now()
print("current time:", ct)

pred_df_regional = vanilla_chronos2_pipeline.predict_df(
    df =df_context.drop(columns=["target_conflict_indicator_local"]), #important to drop the other target, they are strong but not perfect indicators of the other.,
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(),  #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast 
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_regional",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_regional

ct = datetime.datetime.now()
print("current time:", ct)

joblib.dump(vanilla_chronos2_pipeline, f"../Models/Chronos2/chronos2_model_regional_vanilla.pkl") #save the full model as a pickle file. 

current time: 2026-10-10 16:32:13.341084
current time: 2026-10-10 16:32:57.221221


['../Models/Chronos2/chronos2_model_regional_vanilla.pkl']

### Evaluation

In [178]:
eval_df_local = pred_df_local.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_local"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_local

eval_df_regional = pred_df_regional.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_regional"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_regional

,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_regional
0,Central_African_Republic_Bambari,2024-09-26,target_conflict_indicator_regional,0.822698,-0.117672,0.822698,1.012474,0
1,Central_African_Republic_Bambari,2024-10-24,target_conflict_indicator_regional,0.985845,0.274523,0.985845,1.013220,1
2,Central_African_Republic_Bambari,2024-11-21,target_conflict_indicator_regional,0.996836,0.545057,0.996836,1.010762,1
3,Central_African_Republic_Bambari,2024-12-19,target_conflict_indicator_regional,0.999153,0.718722,0.999153,1.007256,0
4,Central_African_Republic_Bambari,2025-01-16,target_conflict_indicator_regional,0.998056,0.800714,0.998056,1.007164,1
...,...,...,...,...,...,...,...,...
1495,Uganda_Zombo_Town_Council,2025-04-10,target_conflict_indicator_regional,0.364299,0.003429,0.364299,1.022263,1
1496,Uganda_Zombo_Town_Council,2025-05-08,target_conflict_indicator_regional,0.374608,-0.003214,0.374608,1.016567,1
1497,Uganda_Zombo_Town_Council,2025-06-05,target_conflict_indicator_regional,0.373668,-0.003334,0.373668,1.012149,1
1498,Uganda_Zombo_Town_Council,2025-07-03,target_conflict_indicator_regional,0.396101,0.008608,0.396101,1.024526,1


### Evalaution: Local Vanilla Model

In [179]:
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_local["target_conflict_indicator_local"].values          # binary 0/1 target
y_prob = eval_df_local["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.98734766
Best threshold (F1): 0.37491074
Best threshold (F2): 0.004167244

===== Metrics for F0.5 (precision-weighted) (threshold=0.987348) =====
Accuracy: 0.9226666666666666
Precision: 0.9866666666666667
Recall: 0.8457142857142858
F1: 0.9107692307692308
ROC AUC: 0.969492857142857
Confusion Matrix:
[[792   8]
 [108 592]]

===== Metrics for F1 (balanced) (threshold=0.374911) =====
Accuracy: 0.926
Precision: 0.9727126805778491
Recall: 0.8657142857142858
F1: 0.9160997732426304
ROC AUC: 0.969492857142857
Confusion Matrix:
[[783  17]
 [ 94 606]]

===== Metrics for F2 (recall-weighted) (threshold=0.004167) =====
Accuracy: 0.8573333333333333
Precision: 0.7780320366132724
Recall: 0.9714285714285714
F1: 0.8640406607369758
ROC AUC: 0.969492857142857
Confusion Matrix:
[[606 194]
 [ 20 680]]


### Evalaution: Regional Vanilla Model

In [180]:

import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_regional["target_conflict_indicator_regional"].values          # binary 0/1 target
y_prob = eval_df_regional["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.9838071
Best threshold (F1): 0.8310252
Best threshold (F2): 1.793541e-05

===== Metrics for F0.5 (precision-weighted) (threshold=0.983807) =====
Accuracy: 0.8826666666666667
Precision: 0.9694323144104804
Recall: 0.8571428571428571
F1: 0.9098360655737705
ROC AUC: 0.9533267873785114
Confusion Matrix:
[[436  28]
 [148 888]]

===== Metrics for F1 (balanced) (threshold=0.831025) =====
Accuracy: 0.9046666666666666
Precision: 0.9339164237123421
Recall: 0.9276061776061776
F1: 0.9307506053268765
ROC AUC: 0.9533267873785114
Confusion Matrix:
[[396  68]
 [ 75 961]]

===== Metrics for F2 (recall-weighted) (threshold=0.000018) =====
Accuracy: 0.8913333333333333
Precision: 0.8646616541353384
Recall: 0.999034749034749
F1: 0.9270040304523063
ROC AUC: 0.9533267873785114
Confusion Matrix:
[[ 302  162]
 [   1 1035]]


## LoRA Training

### Set up the context data
Separate from the previously used context and future datasets

In [181]:
context_ids = list(df_context['id'].unique()) #identify the ids used for context data. 
df_full_lora = df_full[~df_full["id"].isin(context_ids)].copy() #delete the Ids we will test on. 
#cutoffDate = '2024-08-29'
df_context_lora, df_future_lora = build_context_dataset(df = df_full_lora, cutoffDate = '2025-01-01', ids=[],    n_high = 100,     n_low = 50,    n_spike = 50,    n_transition = 50, n_future_obs = 1) #load the datasets: counts are doubled from what we used for building df_context and df_future. 
#del df_future_lora #delete df_future_lora: not required for lora training. 


2016-01-28 00:00:00
2024-12-19 00:00:00
2025-01-16 00:00:00
2025-07-31 00:00:00


## LoRA Training: Local

### Execute LoRA Training: Local

In [182]:
from chronos.chronos2 import preprocess
import datetime

ct = datetime.datetime.now()
print("current time:", ct)


# Prepare data for fine-tuning using the retail sales dataset
train_inputs_lora = preprocess.from_data_frame(
    df_context_lora.drop(columns=["target_conflict_indicator_regional"]), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_local"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)
print("executed train_inputs step")

# Load Chronos‑2
pipeline_lora_local = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model with LoRA
lora_chronos_pipeline_local = pipeline_lora_local.fit(
    inputs=train_inputs_lora,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora"
)
print("executed pipeline_lora.fit step")

joblib.dump(lora_chronos_pipeline_local, f"../Models/Chronos2/chronos2_model_local_lora.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)



current time: 2026-10-10 16:33:02.733453
executed train_inputs step


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
100,0.081077
200,0.064420
300,0.070172
400,0.058606
500,0.085606
600,0.050326
700,0.049222
800,0.041919
900,0.052576
1000,0.039476


executed pipeline_lora.fit step
current time: 2026-10-10 16:57:24.797517


### Local LoRA Model Evaluation

In [183]:
pred_df_local_lora = lora_chronos_pipeline_local.predict_df(
    df =df_context.drop(columns=['target_conflict_indicator_regional']).copy(),
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(), #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_local",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_local_lora

,id,periodStart,target_name,predictions,0.1,0.5,0.9
0,Central_African_Republic_Bambari,2024-09-26,target_conflict_indicator_local,-0.004608,-0.008209,-0.004608,1.138244
1,Central_African_Republic_Bambari,2024-10-24,target_conflict_indicator_local,-0.002844,-0.008358,-0.002844,1.033585
2,Central_African_Republic_Bambari,2024-11-21,target_conflict_indicator_local,-0.000046,-0.007497,-0.000046,1.096576
3,Central_African_Republic_Bambari,2024-12-19,target_conflict_indicator_local,-0.003602,-0.006743,-0.003602,1.134075
4,Central_African_Republic_Bambari,2025-01-16,target_conflict_indicator_local,0.007193,-0.009467,0.007193,1.214450
...,...,...,...,...,...,...,...
1495,Uganda_Zombo_Town_Council,2025-04-10,target_conflict_indicator_local,0.000339,-0.000380,0.000339,0.000751
1496,Uganda_Zombo_Town_Council,2025-05-08,target_conflict_indicator_local,0.000075,-0.000649,0.000075,0.000655
1497,Uganda_Zombo_Town_Council,2025-06-05,target_conflict_indicator_local,0.000243,-0.000633,0.000243,0.000952
1498,Uganda_Zombo_Town_Council,2025-07-03,target_conflict_indicator_local,0.000225,-0.000498,0.000225,0.000716


In [184]:
eval_df_local_lora = pred_df_local_lora.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_local"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_local_lora


,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_local
0,Central_African_Republic_Bambari,2024-09-26,target_conflict_indicator_local,-0.004608,-0.008209,-0.004608,1.138244,1
1,Central_African_Republic_Bambari,2024-10-24,target_conflict_indicator_local,-0.002844,-0.008358,-0.002844,1.033585,0
2,Central_African_Republic_Bambari,2024-11-21,target_conflict_indicator_local,-0.000046,-0.007497,-0.000046,1.096576,0
3,Central_African_Republic_Bambari,2024-12-19,target_conflict_indicator_local,-0.003602,-0.006743,-0.003602,1.134075,0
4,Central_African_Republic_Bambari,2025-01-16,target_conflict_indicator_local,0.007193,-0.009467,0.007193,1.214450,0
...,...,...,...,...,...,...,...,...
1495,Uganda_Zombo_Town_Council,2025-04-10,target_conflict_indicator_local,0.000339,-0.000380,0.000339,0.000751,0
1496,Uganda_Zombo_Town_Council,2025-05-08,target_conflict_indicator_local,0.000075,-0.000649,0.000075,0.000655,0
1497,Uganda_Zombo_Town_Council,2025-06-05,target_conflict_indicator_local,0.000243,-0.000633,0.000243,0.000952,0
1498,Uganda_Zombo_Town_Council,2025-07-03,target_conflict_indicator_local,0.000225,-0.000498,0.000225,0.000716,0


In [185]:
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_local_lora["target_conflict_indicator_local"].values          # binary 0/1 target
y_prob = eval_df_local_lora["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.9920836
Best threshold (F1): 0.47654283
Best threshold (F2): 0.0013437793

===== Metrics for F0.5 (precision-weighted) (threshold=0.992084) =====
Accuracy: 0.9226666666666666
Precision: 0.9866666666666667
Recall: 0.8457142857142858
F1: 0.9107692307692308
ROC AUC: 0.9529392857142858
Confusion Matrix:
[[792   8]
 [108 592]]

===== Metrics for F1 (balanced) (threshold=0.476543) =====
Accuracy: 0.9206666666666666
Precision: 0.9574803149606299
Recall: 0.8685714285714285
F1: 0.9108614232209737
ROC AUC: 0.9529392857142858
Confusion Matrix:
[[773  27]
 [ 92 608]]

===== Metrics for F2 (recall-weighted) (threshold=0.001344) =====
Accuracy: 0.8513333333333334
Precision: 0.7744533947065593
Recall: 0.9614285714285714
F1: 0.8578712555768006
ROC AUC: 0.9529392857142858
Confusion Matrix:
[[604 196]
 [ 27 673]]


### Execute LoRA Training: Regional

In [186]:
from chronos.chronos2 import preprocess
import datetime

ct = datetime.datetime.now()
print("current time:", ct)


# Prepare data for fine-tuning using the retail sales dataset
train_inputs_lora = preprocess.from_data_frame(
    df_context_lora.drop(columns=["target_conflict_indicator_local"]).copy(), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_regional"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)
print("executed train_inputs step")

# Load Chronos‑2
pipeline_lora_regional = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model with LoRA
lora_chronos_pipeline_regional = pipeline_lora_regional.fit(
    inputs=train_inputs_lora,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-4,
    batch_size=32,
    logging_steps=100,
    finetune_mode="lora"
)
print("executed pipeline_lora.fit step")

joblib.dump(lora_chronos_pipeline_regional, f"../Models/Chronos2/chronos2_model_regional_lora.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)


current time: 2026-10-10 17:00:01.855238
executed train_inputs step


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
100,0.058511
200,0.048238
300,0.053042
400,0.055944
500,0.041428
600,0.049971
700,0.052792
800,0.042943
900,0.052226
1000,0.048731


executed pipeline_lora.fit step
current time: 2026-10-10 17:24:15.051586


In [187]:
pred_df_regional_lora = lora_chronos_pipeline_regional.predict_df(
    df = df_context.drop(columns=["target_conflict_indicator_local"]).copy(), #important to drop the other target, they are strong but not perfect indicators of the other. 
    future_df=df_future.drop(columns=['target_conflict_indicator_local', 'target_conflict_indicator_regional']).copy(), #drop the target cols from the future df; we don't need them. 
    prediction_length=12,  # Number of steps to forecast
    quantile_levels=[0.1, 0.5, 0.9],  # Quantiles for probabilistic forecast; 3x answers per test, model is 10%, 50% and 90% confident the value is below this answer. 
    id_column="id",  # Column identifying different time series
    timestamp_column="periodStart",  # Column with datetime information
    target="target_conflict_indicator_regional",  # Column(s) with time series values to predict
    cross_learning=True,  # Enable cross-learning;
)
pred_df_regional_lora

,id,periodStart,target_name,predictions,0.1,0.5,0.9
0,Central_African_Republic_Bambari,2024-09-26,target_conflict_indicator_regional,0.988524,-0.279576,0.988524,1.012939
1,Central_African_Republic_Bambari,2024-10-24,target_conflict_indicator_regional,1.006905,-0.257615,1.006905,1.009918
2,Central_African_Republic_Bambari,2024-11-21,target_conflict_indicator_regional,1.001479,-0.024952,1.001479,1.006418
3,Central_African_Republic_Bambari,2024-12-19,target_conflict_indicator_regional,0.997739,0.038349,0.997739,1.005608
4,Central_African_Republic_Bambari,2025-01-16,target_conflict_indicator_regional,0.992306,0.095227,0.992306,1.002183
...,...,...,...,...,...,...,...
1495,Uganda_Zombo_Town_Council,2025-04-10,target_conflict_indicator_regional,0.619834,-0.025017,0.619834,1.002558
1496,Uganda_Zombo_Town_Council,2025-05-08,target_conflict_indicator_regional,0.689605,-0.030046,0.689605,0.987534
1497,Uganda_Zombo_Town_Council,2025-06-05,target_conflict_indicator_regional,0.584244,-0.010065,0.584244,1.007059
1498,Uganda_Zombo_Town_Council,2025-07-03,target_conflict_indicator_regional,0.586585,-0.015592,0.586585,0.998542


### Regional LoRA Model Evaluation

In [188]:
eval_df_regional_lora = pred_df_regional_lora.merge(
    df_future[["id", "periodStart", "target_conflict_indicator_regional"]],
    on=["id", "periodStart"],
    how="inner"
)
eval_df_regional_lora

,id,periodStart,target_name,predictions,0.1,0.5,0.9,target_conflict_indicator_regional
0,Central_African_Republic_Bambari,2024-09-26,target_conflict_indicator_regional,0.988524,-0.279576,0.988524,1.012939,0
1,Central_African_Republic_Bambari,2024-10-24,target_conflict_indicator_regional,1.006905,-0.257615,1.006905,1.009918,1
2,Central_African_Republic_Bambari,2024-11-21,target_conflict_indicator_regional,1.001479,-0.024952,1.001479,1.006418,1
3,Central_African_Republic_Bambari,2024-12-19,target_conflict_indicator_regional,0.997739,0.038349,0.997739,1.005608,0
4,Central_African_Republic_Bambari,2025-01-16,target_conflict_indicator_regional,0.992306,0.095227,0.992306,1.002183,1
...,...,...,...,...,...,...,...,...
1495,Uganda_Zombo_Town_Council,2025-04-10,target_conflict_indicator_regional,0.619834,-0.025017,0.619834,1.002558,1
1496,Uganda_Zombo_Town_Council,2025-05-08,target_conflict_indicator_regional,0.689605,-0.030046,0.689605,0.987534,1
1497,Uganda_Zombo_Town_Council,2025-06-05,target_conflict_indicator_regional,0.584244,-0.010065,0.584244,1.007059,1
1498,Uganda_Zombo_Town_Council,2025-07-03,target_conflict_indicator_regional,0.586585,-0.015592,0.586585,0.998542,1


In [189]:
import numpy as np
from sklearn.metrics import (
    precision_recall_curve, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score
)

# ============================
# 1. Extract true labels + probabilities
# ============================

y_true = eval_df_regional_lora["target_conflict_indicator_regional"].values          # binary 0/1 target
y_prob = eval_df_regional_lora["0.5"].values                       # Chronos2 median probability


# ============================
# 2. Compute precision-recall curve
# ============================

precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

# Align lengths: thresholds is length N-1
precision = precision[:-1]
recall = recall[:-1]


# ============================
# 3. Compute F0.5, F1, F2 scores
# ============================

# F0.5: precision weighted twice as much as recall
f05_scores = (1.25 * precision * recall) / (0.25 * precision + recall + 1e-9)

# F1: balanced
f1_scores_curve = (2 * precision * recall) / (precision + recall + 1e-9)

# F2: recall weighted twice as much as precision
f2_scores = (5 * precision * recall) / (4 * precision + recall + 1e-9)


# ============================
# 4. Select best thresholds
# ============================

best_idx_f05 = np.argmax(f05_scores)
best_threshold_f05 = thresholds[best_idx_f05]

best_idx_f1 = np.argmax(f1_scores_curve)
best_threshold_f1 = thresholds[best_idx_f1]

best_idx_f2 = np.argmax(f2_scores)
best_threshold_f2 = thresholds[best_idx_f2]


print("Best threshold (F0.5):", best_threshold_f05)
print("Best threshold (F1):", best_threshold_f1)
print("Best threshold (F2):", best_threshold_f2)


# ============================
# 5. Function to compute metrics for a given threshold
# ============================

def compute_metrics(threshold, name):
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n===== Metrics for {name} (threshold={threshold:.6f}) =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_prob))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


# ============================
# 6. Output metrics for each threshold
# ============================

compute_metrics(best_threshold_f05, "F0.5 (precision-weighted)")
compute_metrics(best_threshold_f1, "F1 (balanced)")
compute_metrics(best_threshold_f2, "F2 (recall-weighted)")


Best threshold (F0.5): 0.9903331
Best threshold (F1): 0.17958492
Best threshold (F2): 0.0019975104

===== Metrics for F0.5 (precision-weighted) (threshold=0.990333) =====
Accuracy: 0.8626666666666667
Precision: 0.9510869565217391
Recall: 0.8445945945945946
F1: 0.8946830265848671
ROC AUC: 0.91866512448409
Confusion Matrix:
[[419  45]
 [161 875]]

===== Metrics for F1 (balanced) (threshold=0.179585) =====
Accuracy: 0.8953333333333333
Precision: 0.8721422523285352
Recall: 0.9942084942084942
F1: 0.9291835814163284
ROC AUC: 0.91866512448409
Confusion Matrix:
[[ 313  151]
 [   6 1030]]

===== Metrics for F2 (recall-weighted) (threshold=0.001998) =====
Accuracy: 0.8933333333333333
Precision: 0.8662207357859532
Recall: 1.0
F1: 0.9283154121863799
ROC AUC: 0.91866512448409
Confusion Matrix:
[[ 304  160]
 [   0 1036]]


## Full Training: Local
Started at 1930

### Build the Dataset

In [18]:
future_keys = df_future[["name", "periodStart"]].drop_duplicates()
df_future_fixed = df_full.merge(future_keys, on=["name", "periodStart"], how="inner")

In [19]:
df_context.head(5)

,id,name,country,latitude,longitude,minBorderDistanceKm,minCapitalDistanceKm,periodStart,LocalEventCount,LocalDistinctActorCount,...,Renewable_internal_freshwater_resources_per_capita_cubic_meters_ERH2OINTRPC,Cereal_yield_kg_per_hectare_AGYLDCRELKG,Permanent_cropland__of_land_area_AGLNDCROPZS,Water_productivity_total_constant_2015_US_GDP_per_cubic_meter_of_total_freshwater_withdrawal_ERGDPFWTLM3KD,Internally_displaced_persons_new_displacement_associated_with_conflict_and_violence_number_of_cases_VCIDPNWCV,is_voting,target_conflict_indicator_local,target_conflict_indicator_regional,conflict_score,transition_flag
49,Central_African_Republic_Bambari,Bambari,Central African Republic,6.1389,20.777901,155.318741,313.252533,2016-01-28,0,0,...,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,1,1,0
16685,Central_African_Republic_Bambari,Bambari,Central African Republic,6.1389,20.777901,155.318741,313.252533,2016-02-25,4,5,...,29913.042969,875.099976,0.203859,24.501831,46000.0,False,1,1,2,1
33321,Central_African_Republic_Bambari,Bambari,Central African Republic,6.1389,20.777901,155.318741,313.252533,2016-03-24,5,6,...,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0,0,1
49957,Central_African_Republic_Bambari,Bambari,Central African Republic,6.1389,20.777901,155.318741,313.252533,2016-04-21,2,3,...,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,1,1,1
66593,Central_African_Republic_Bambari,Bambari,Central African Republic,6.1389,20.777901,155.318741,313.252533,2016-05-19,0,0,...,29913.042969,875.099976,0.203859,24.501831,46000.0,False,0,0,0,1


In [20]:
from chronos.chronos2 import preprocess
import datetime

ct = datetime.datetime.now()
print("current time:", ct)

#generate validation inputes in the same format as the training inputs (we do this from df_future_lora; we are not LoRA training here!)
# Prepare data for fine-tuning using the retail sales dataset
training_inputs_full_ft = preprocess.from_data_frame(
    df_context.drop(columns=["target_conflict_indicator_regional"]), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_local"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)

validation_inputs_full_ft = preprocess.from_data_frame(
    df_future.drop(columns=["target_conflict_indicator_regional"]), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_local"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)





current time: 2026-10-10 19:12:12.261659


In [22]:
from chronos import Chronos2Pipeline

In [ ]:
# Load Chronos‑2
pipeline_full_finetuning_local = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model (full)
full_finetining_chronos_pipeline_local = pipeline_full_finetuning_local.fit(
    inputs=training_inputs_full_ft,
  #  validation_inputs= validation_inputs_full_ft,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-6,
    batch_size=256, #256 default
    logging_steps=25,
    finetune_mode="full",
    lr_scheduler_type="linear",
)
print("executed pipeline_full_finetuning.fit step")

joblib.dump(full_finetining_chronos_pipeline_local, f"../Models/Chronos2/chronos2_model_local_full_finetuning.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)

gc.collect()

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Step,Training Loss
25,0.034850


### Full Fine Tuning: Regional

In [ ]:
from chronos.chronos2 import preprocess
import datetime

ct = datetime.datetime.now()
print("current time:", ct)

#generate validation inputes in the same format as the training inputs (we do this from df_future_lora; we are not LoRA training here!)
# Prepare data for fine-tuning using the retail sales dataset
training_inputs_full_ft = preprocess.from_data_frame(
    df_context.drop(columns=["target_conflict_indicator_local"]), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_regional"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)

validation_inputs_full_ft = preprocess.from_data_frame(
    df_future.drop(columns=["target_conflict_indicator_local"]), #important to drop the other target, they are strong but not perfect indicators of the other. 
    target_columns=["target_conflict_indicator_regional"],
    prediction_length=12,
    id_column="id",
    timestamp_column="periodStart",
    known_covariates_names=[ "name",	"country",	"latitude",	"longitude",	"minBorderDistanceKm",	"minCapitalDistanceKm", "is_voting"], #Knowable values in the future dataset. Remaining columns are treated as past-only covariates
)





In [ ]:
# Load Chronos‑2
pipeline_full_finetuning_local = Chronos2Pipeline.from_pretrained(
    "amazon/chronos-2",
    device_map="cpu"   #"CUDA also an option if you're a baller with a big GPU (I am not). 
)

# Fine-tune the model (full)
full_finetining_chronos_pipeline_local = pipeline_full_finetuning_local.fit(
    inputs=training_inputs_full_ft,
  #  validation_inputs= validation_inputs_full_ft,
    prediction_length=12, #must match the number of observations in the future_df per place. 
    num_steps=1000,#Set to 10 just to check that it works. 
    learning_rate=1e-6,
    batch_size=256, #256 default
    logging_steps=25,
    finetune_mode="full",
    lr_scheduler_type="linear",
)
print("executed pipeline_full_finetuning.fit step")

joblib.dump(full_finetining_chronos_pipeline_local, f"../Models/Chronos2/chronos2_model_regional_full_finetuning.pkl") #save the full model as a pickle file. 


ct = datetime.datetime.now()
print("current time:", ct)
